<a href="https://colab.research.google.com/github/pierrebg18/tp-information-retrieval-with-llm-student-version/blob/main/1-Recherche%20d'information%20classique.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Partie 1. - Recherche d'Information classique

Dans cette partie, nous allons mettre en œuvre des principes et modèles classiques de Recherche d'Information. Le jeu de données est une collection de livres au format texte (.txt) de Henry Rider Haggard. Jetez un œil à ces documents dans le dossier _data_.

En sortie de ce module, vous serez capable de :

- Construire un index inversé ;
- Effectuer des requêtes simples selon le modèle booléen :
- Calculer la pondération des termes selon la méthode TF-IDF ;
- Mettre en œuvre le modèle vectoriel de recherche d'information et y appliquer des requêtes.

### Import des bibliothèques logicielles et configuration

Les lignes suivantes permettent d'instancier un objet de la classe `IRSystem` représentant notre moteur de recherche et de charger les données en RAM.

In [26]:
import sys

# Vérifie si le code est exécuté sur Google Colab
if 'google.colab' in sys.modules:
    # Commandes à exécuter uniquement sur Google Colab
    !git clone https://github.com/vincentmartin/tp-information-retrieval-with-llm-student-version.git
    %cd tp-information-retrieval-with-llm-student-version
else:
    # Commandes à exécuter si ce n'est pas sur Google Colab
    print("Pas sur Google Colab, ces commandes ne seront pas exécutées.")

Cloning into 'tp-information-retrieval-with-llm-student-version'...
remote: Enumerating objects: 2262, done.
remote: Counting objects: 100% (50/50), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 2262 (delta 30), reused 37 (delta 19), pack-reused 2212 (from 1)
Receiving objects: 100% (2262/2262), 13.92 MiB | 13.83 MiB/s, done.
Resolving deltas: 100% (32/32), done.
/content/tp-information-retrieval-with-llm-student-version/tp-information-retrieval-with-llm-student-version/tp-information-retrieval-with-llm-student-version/tp-information-retrieval-with-llm-student-version


#### Chargement des données

Les lignes ci-dessous permettent de charger les données qui sont un ensemble de 60 livres au format texte (.txt) d'[Henry Rider Haggard](https://fr.wikipedia.org/wiki/Henry_Rider_Haggard).


In [27]:
from classic_ir.IRSystem import *

# !rm -rf ./data/RiderHaggard/stemmed
ir_system = IRSystem()
ir_system.read_data('./data/RiderHaggard') # chargement des données et prétraitement des documents (stemming).

Reading in documents...
Stemming Documents...
Nada the Lily 1207.txt
    Doc 1 of 60: Nada the Lily
The Witch's Head (1884) 0500791.txt
    Doc 2 of 60: The Witch's Head (1884)
The Ancient Allan 5746.txt
    Doc 3 of 60: The Ancient Allan
Beatrice 3096.txt
    Doc 4 of 60: Beatrice
Ayesha, the Return of She 5228.txt
    Doc 5 of 60: Ayesha, the Return of She
The Mahatma and the Hare 2764.txt
    Doc 6 of 60: The Mahatma and the Hare
She 3155.txt
    Doc 7 of 60: She
The People of the Mist 6769.txt
    Doc 8 of 60: The People of the Mist
Smith and the Pharaohs, and other Tales 6073.txt
    Doc 9 of 60: Smith and the Pharaohs, and other Tales
Finished 1724.txt
    Doc 10 of 60: Finished
The Brethren 2762.txt
    Doc 11 of 60: The Brethren
Wisdom's Daughter (1923) 0200181.txt
    Doc 12 of 60: Wisdom's Daughter (1923)
She and Allan 5745.txt
    Doc 13 of 60: She and Allan
Swallow a tale of the great trek 4074.txt
    Doc 14 of 60: Swallow a tale of the great trek
Fair Margaret 9780.txt
  

### Exercice 1. - Construction de l'index inversé

Ce premier exercice a pour objectif de construire l'index inversé non positionnel. L'attribut `self.inverted_index` est un tableau associatif qui associe une liste d'entiers (docIDs) à un mot (word).

Documentation ici https://docs.python.org/3/library/collections.html#collections.defaultdict.

Exercice : modifier la fonction `index` pour calculer l'index inversé.

**Indication** : profitez du parcours des documents pour remplir aussi `self.tf`, qui associe à chaque docID un `Counter` donnant le nombre d'occurrences de chaque mot dans ce document. Vous en aurez besoin à l'exercice 3 pour calculer les poids TF-IDF.

Le résultat ci-dessous indique que vous avez réussi.
```
===== Running tests =====
Inverted Index Test
    Score: 3 Feedback: 5/5 Correct. Accuracy: 1.000000
```

In [28]:
print(dir(ir_system))
print(ir_system.vocab)

['_IRSystem__read_raw_data', '_IRSystem__read_stemmed_data', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__firstlineno__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__static_attributes__', '__str__', '__subclasshook__', '__weakref__', 'alphanum', 'boolean_retrieve', 'compute_tfidf', 'docs', 'get_posting', 'get_posting_unstemmed', 'get_tfidf', 'get_tfidf_unstemmed', 'get_uniq_words', 'index', 'p', 'process_query', 'query_rank', 'query_retrieve', 'rank_retrieve', 'read_data', 'titles', 'vocab']
['meannot', 'stewardship', 'manur', 'rica', 'sixmil', 'numberthat', 'op', 'ayesha', 'bullhippopotamu', 'twoandthirti', 'stableentr', 'somhlolo', 'moodin', 'ia', 'waysy', 'matutin', 'seafrontag', 'trouver', 'littleand', 'evict', 'lostal', 'sire', 'canari', 'halfsist', 'twi

In [29]:
# Exercice 1. Indexation
from collections import Counter


def index(self):
    """
    Construit l'index inversé et le stocke dans self.inverted_index.

    Dans le code ci-dessous, seul le dictionnaire des tokens est construit. Les postings lists sont vides.
    """

    print("Indexing...")

    # initialisation
    self.tf = defaultdict(Counter)
    inverted_index = defaultdict(list)
    for word in self.vocab:
        inverted_index[word] = list()

    #calcule du counter de chaque doc et index inversé
    for doc_id in range(len(self.docs)):
        doc = self.docs[doc_id]
        self.tf[doc_id] = Counter(doc) # calcul conter de chaque doc
        for word in self.tf[doc_id]: # prend le counter et ajout l'index inversé
            inverted_index[word].append(doc_id)

    self.inverted_index = inverted_index



# Ne pas modifier les lignes ci-dessous
IRSystem.index = index
ir_system.index()
run_tests(ir_system, part=0)

Indexing...
===== Running tests =====
Inverted Index Test
    Score: 3 Feedback: 5/5 Correct. Accuracy: 1.000000


### Exercice 2. - Recherche booléenne

Ce deuxième exercice a pour objectif d'implémenter la recherche booléenne. La requête `query` est une liste de mots _racinisés_ (stemming de Porter) et l'algorithme doit rechercher les documents qui contiennent TOUS ces mots.


Exercice : modifier la fonction `boolean_retrieve` pour implémenter la recherche booléenne.


Le résultat ci-dessous indique que vous avez réussi.
```
===== Running tests =====
Boolean Retrieval Test
    Score: 3 Feedback: 5/5 Correct. Accuracy: 1.000000
```

In [30]:
# Exercice 2. Recherche booléenne
def boolean_retrieve(self, query):
    """
    A partir d'une requête sous la forme d'une liste de mots *racinisés*,
    retourne la liste des documents dans lesquels *tous* ces mots apparaissent (ie une requête AND).
    Retourne une liste vide si la requête ne retourne aucun document.

    Dans le code ci-dessous, tous les documents répondent.
    """
    docs = list()
    tab= self.inverted_index[query[0]]
    for mot in query[1:]:
      tab=set(tab)&set(self.inverted_index[mot]) # fait un and entre deux list
    return tab

# Ne pas modifier les lignes ci-dessous
IRSystem.boolean_retrieve = boolean_retrieve
run_tests(ir_system, part=1)

===== Running tests =====
Boolean Retrieval Test
    Score: 3 Feedback: 5/5 Correct. Accuracy: 1.000000


### Exercice 3. - Calcul des poids TF-IDF des termes dans les documents

Dans ce troisième exercice, l'objectif est de pré-calculer les poids TF-IDF pour chaque terme dans chaque document. Pour ce faire, appliquer la formule vue en cours. Utiliser le logarithme en base 10.


Exercice : modifier la fonction `compute_tfidf` pour calculer les poids TF-IDF.

Le résultat ci-dessous indique que vous avez réussi.
```
Calculating tf-idf...
===== Running tests =====
TF-IDF Test
    Score: 3 Feedback: 5/5 Correct. Accuracy: 1.000000
```

In [33]:
# Exercice 3. calcul des scores tf-idf

import math
def compute_tfidf(self):
    """
    Calcule les scores tf-idf pour tous les mots de tous les documents et les stocke dans self.tfidf.

    Dans le code ci-dessous, les scores tf-idf sont tous nuls.
    """
    print("Calculating tf-idf...")

    self.tfidf = defaultdict(Counter)
    idf = {}



    ### Calcul de l'IDF ###
    N = len(self.docs)  # N = nombre de documents
    for word in self.vocab:
        df=len(self.inverted_index[word])
        idf[word] = math.log10(N / df)
    #########################


    ### Calcul du TF-IDF ###
    for i in range(N):
        counts = Counter(self.docs[i])  # fréquence de chaque mot dans le document i
        for word in self.vocab:
            if counts[word] > 0:
                tf = 1 + math.log10(counts[word])
                self.tfidf[i][word] = tf * idf[word]
            else:
                self.tfidf[i][word] = 0.

# Ne pas modifier les lignes ci-dessous
IRSystem.compute_tfidf = compute_tfidf
ir_system.compute_tfidf()
run_tests(ir_system, part=2)

Calculating tf-idf...
===== Running tests =====
TF-IDF Test
    Score: 3 Feedback: 5/5 Correct. Accuracy: 1.000000


#### Question - Du bruit dans le corpus ?

Les livres proviennent du [Projet Gutenberg](https://www.gutenberg.org/). Ouvrez quelques fichiers du dossier `data/RiderHaggard/raw` et regardez leur dernière ligne : la plupart se terminent par une mention ajoutée par le Projet Gutenberg, du type « End of the Project Gutenberg EBook of She, by H. Rider Haggard ».

1. Affichez, pour les termes `gutenberg`, `ebook` et `project`, le nombre de documents qui les contiennent (df) et leur idf. Rappel : `self.inverted_index` attend des mots racinisés, utilisez `get_posting_unstemmed`.
2. Pourquoi le poids TF-IDF de `gutenberg` reste-t-il faible dans tous les documents, alors que ce mot n'a rien à voir avec les romans ?
3. Quel serait l'idf de `gutenberg` si un seul livre contenait cette mention ? Quel risque cela poserait-il pour la recherche ?
4. Quel traitement proposeriez-vous avant l'indexation ? Faites le lien avec la partie du cours « Représenter les documents ».

In [39]:
# Question - Du bruit dans le corpus ?
N = len(ir_system.docs)
for word in ['gutenberg', 'ebook', 'project']:
    # A COMPLETER : afficher df et idf du mot
    posting = ir_system.get_posting_unstemmed(word)
    df = len(set(posting))
    idf=math.log10(N/df)
    print(word+" df nbr :"+str(df))
    print(word+" idf nbr :"+str(idf))



# Vos réponses aux questions 2 à 4 :
# 2.Car il est sité dans tout les livre ce qui fait baisser son importante (il ne s'agit pas d'un terme rare) la forme tf*idf le considère comme un mot inutile de sens
# 3.Son IDF serait égal a N cad très élèves car ce mot aurait une forte importance dans un recherche. Le risque est que ce document ecrase l'importance des autres mot de la requete
# 4.formatter les mots (enlever majuscule etc ...) , enlever les mots vides , enlever les mots extremes qui on un df trop élévé ou trop faible car peuvent etre parasite


gutenberg df nbr :52
gutenberg idf nbr :0.062147906748844434
ebook df nbr :38
ebook idf nbr :0.19836765376683346
project df nbr :60
project idf nbr :0.0


### Exercice 4. - Calcul de la similarité cosinus.

Dans ce quatrième exercice, l'objectif est de calculer la similarité cosinus entre le vecteur requête `query` et chaque vecteur document. Pour ce faire, appliquer la formule vue en cours en considérant les éléments suivants :
- **Vecteurs documents** : poids TF-IDF calculés à l'exercice 3, **normalisés L2** (on divise par la norme du vecteur document).
- **Vecteur requête** : poids TF logarithmique `1 + log10(tf)`, **sans IDF et sans normalisation**.
- Utiliser le logarithme en base 10.

Le score attendu est donc :

$$score(q, d) = \frac{\sum_{t \in q} w_{t,q} \cdot w_{t,d}}{\lVert d \rVert_2}$$

Ne pas normaliser le vecteur requête ne change pas le classement (la norme de la requête est la même pour tous les documents), mais change la valeur du score, qui est vérifiée par le test. En notation SMART, il s'agit du schéma **ltc.lnn**.

Exercice : modifier la fonction `rank_retrieve` pour implémenter la similarité cosinus.

Le résultat ci-dessous indique que vous avez réussi.
```
===== Running tests =====
Cosine Similarity Test
    Score: 3 Feedback: 5/5 Correct. Accuracy: 1.000000
```

In [42]:
# Exercice 4. Similarité cosinus
def rank_retrieve(self, query):
    """
    A partir d'une requête (une liste de mots), retourne la liste des 10 meilleurs documents (docID, score), classés par score décroissant, en appliquant la similarité cosinus.

    Dans l'exemple ci-dessous. C'est la mesure de Jaccard qui est utilisée.
    """
    scores = [0.0 for _ in range(len(self.docs))]

    query_set = set(query)


    #som=som+(self.tfidf[doc_set][mot_query]*self.tfidf[doc_set][mot_query])

    for d in range(len(self.docs)):
        doc_set = set(self.docs[d])
        som=0
        for mot_query in query_set:
          # poids de la requête : 1 + log10(tf), sans IDF
          w_q = 1 + math.log10(query.count(mot_query))
          # poids TF-IDF du document (exercice 3)
          som=som+(w_q*self.tfidf[d].get(mot_query, 0.0)) #get renvoie soit le mot soit 0

        somme_carres = 0.0
        for mot in self.tfidf[d]:
            w = self.tfidf[d][mot]
            somme_carres = somme_carres + w * w
        norme = math.sqrt(somme_carres)

        # score = produit scalaire / norme du document
        if norme != 0:
            scores[d] = som / norme
        else:
            scores[d] = 0.0


    # Tri des scores
    ranking = [idx for idx, sim in sorted(enumerate(scores),
                                        key=lambda pair: pair[1], reverse=True)]
    results = []
    for i in range(10):
        results.append((ranking[i], scores[ranking[i]]))
    return results

# Ne pas modifier les lignes ci-dessous
IRSystem.rank_retrieve = rank_retrieve
run_tests(ir_system, part=3)

===== Running tests =====
Cosine Similarity Test
    Score: 3 Feedback: 5/5 Correct. Accuracy: 1.000000
